<a href="https://colab.research.google.com/github/2001lida/PythonLession2/blob/hw_8/%D0%97%D0%B0%D0%B4%D0%B0%D0%BD%D0%B8%D0%B58.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import requests
import pandas as pd
import psycopg2

BASE_URL = "https://www.anapioficeandfire.com/api"

# -----------------------------
# 1. Получение всех книг
# -----------------------------
books_response = requests.get(f"{BASE_URL}/books")
books = books_response.json()
df_books = pd.DataFrame(books)

# -----------------------------
# 2. Получение всех домов (с пагинацией)
# -----------------------------
def fetch_all_houses(params=None):
    if params is None:
        params = {}

    all_data = []
    page = 1

    while True:
        query = {
            "page": page,
            "pageSize": 50,
            **params
        }

        response = requests.get(f"{BASE_URL}/houses", params=query)
        data = response.json()

        if not data:
            break

        all_data.extend(data)
        page += 1

    return pd.DataFrame(all_data)


# все дома
df_houses = fetch_all_houses()

# -----------------------------
# 3. Дома с девизом (hasWords)
# -----------------------------
df_houses_with_words = fetch_all_houses({"hasWords": "true"})

# дополнительная фильтрация (на всякий случай)
df_houses_with_words = df_houses_with_words[
    df_houses_with_words["words"].astype(str).str.strip() != ""
]


print("Books:", df_books.shape)
print("Houses:", df_houses.shape)
print("Houses with words:", df_houses_with_words.shape)

# пример
df_books_clean = df_books.copy()

df_books_clean["authors"] = df_books_clean["authors"].apply(lambda x: ", ".join(x))
df_books_clean["released"] = pd.to_datetime(df_books_clean["released"])

print(df_books_clean[["name", "authors", "numberOfPages", "released"]])

df_houses_clean = df_houses.copy()

# списки → строки
for col in ["titles", "seats"]:
    df_houses_clean[col] = df_houses_clean[col].apply(lambda x: ", ".join(x) if x else "")

# убрать пустые названия домов
df_houses_clean = df_houses_clean[df_houses_clean["name"] != ""]

print(df_houses_clean[["name", "region", "words"]].head())

df_houses_words_clean = df_houses_with_words.copy()

df_houses_words_clean = df_houses_words_clean[
    df_houses_words_clean["words"].str.strip() != ""
]

print(df_houses_words_clean[["name", "region", "words"]])


# -----------------------------
# 1. Подключение к БД
# -----------------------------
conn = psycopg2.connect(
    host="hh-pgsql-public.ebi.ac.uk",
    database="pfmegrnargs",
    user="reader",
    password="NWDMCE5xdipIjRrp"
)

cursor = conn.cursor()

# -----------------------------
# 2. Получить 10 строк (все столбцы)
# -----------------------------
cursor.execute("SELECT * FROM rnc_database LIMIT 10")
rows = cursor.fetchall()

# названия колонок
columns = [desc[0] for desc in cursor.description]

df_all = pd.DataFrame(rows, columns=columns)

print("Все столбцы:")
print(df_all.head())

# -----------------------------
# 3. Получить нужные столбцы
# -----------------------------
query = """
SELECT display_name, num_sequences, num_organisms, url
FROM rnc_database
LIMIT 10
"""

cursor.execute(query)
rows_selected = cursor.fetchall()

columns_selected = [desc[0] for desc in cursor.description]

df_selected = pd.DataFrame(rows_selected, columns=columns_selected)

print("\nТолько нужные столбцы:")
print(df_selected.head())

# -----------------------------
# 4. Закрытие соединения
# -----------------------------
cursor.close()
conn.close()

Books: (10, 11)
Houses: (444, 16)
Houses with words: (68, 16)
                         name              authors  numberOfPages   released
0           A Game of Thrones  George R. R. Martin            694 1996-08-01
1            A Clash of Kings  George R. R. Martin            768 1999-02-02
2           A Storm of Swords  George R. R. Martin            992 2000-10-31
3            The Hedge Knight  George R. R. Martin            164 2005-03-09
4           A Feast for Crows  George R. R. Martin            784 2005-11-08
5             The Sworn Sword  George R. R. Martin            152 2008-06-18
6          The Mystery Knight  George R. R. Martin            416 2011-03-29
7        A Dance with Dragons  George R. R. Martin           1040 2011-07-12
8  The Princess and the Queen  George R. R. Martin            784 2013-12-03
9            The Rogue Prince  George R. R. Martin            832 2014-06-17
                          name           region            words
0                 House Al